# **Feature Engineering**

**Add the cleaned dataset**

In [1]:
import pandas as pd

hourly_demand = pd.read_csv('/kaggle/input/datasets/sanyazuberi/ev-charging-cleaned-hourly-and-daily/hourly_demand.csv', index_col=0, parse_dates=True)
daily_demand = pd.read_csv('/kaggle/input/datasets/sanyazuberi/ev-charging-cleaned-hourly-and-daily/daily_demand.csv', index_col=0, parse_dates=True)

**Calender features**  
Hour of day, day of week, month, and weekend flag were extracted directly from the timestamp index, giving the models explicit access to the daily/weekly patterns identified during EDA.

In [2]:
hourly_demand['hour'] = hourly_demand.index.hour
hourly_demand['day_of_week'] = hourly_demand.index.dayofweek
hourly_demand['month'] = hourly_demand.index.month
hourly_demand['is_weekend'] = hourly_demand['day_of_week'].isin([5,6]).astype(int)

**Lag features**  
For both session count and energy, lag features were added for the values from 1 hour ago, 24 hours ago (same hour, previous day), and 168 hours ago (same hour, one week prior). I included the 168-hour lag specifically because the EDA showed a clear weekly pattern, making last week’s same-hour value a plausible predictor. These features give tree-based models like XGBoost, which have no inherent sense of time order, a way to access recent history.

In [3]:
# lag features for session_count
hourly_demand['lag_1h_session_count'] = hourly_demand['session_count'].shift(1)
hourly_demand['lag_24h_session_count'] = hourly_demand['session_count'].shift(24)
hourly_demand['lag_168h_session_count'] = hourly_demand['session_count'].shift(168)

# lag features for total_energy_kwh
hourly_demand['lag_1h_energy'] = hourly_demand['total_energy_kwh'].shift(1)
hourly_demand['lag_24h_energy'] = hourly_demand['total_energy_kwh'].shift(24)
hourly_demand['lag_168h_energy'] = hourly_demand['total_energy_kwh'].shift(168)

**Rolling averages**  
A 7-day (168-hour) rolling average was added for targets, giving the model a smoothed sense of the recent overall demand level, in addition to single point-in-time lag values.
(Note: the first 168 hours of the dataset have missing values in these engineered columns, since there isn’t enough prior history to compute them. This is expected and handled during the train/test split.)


In [4]:
# rolling averages for session_count
hourly_demand['rolling_7d_avg_session_count'] = hourly_demand['session_count'].rolling(window=168).mean()

# rolling averages for total_energy_kwh
hourly_demand['rolling_7d_avg_energy'] = hourly_demand['total_energy_kwh'].rolling(window=168).mean()

hourly_demand.isna().sum()

session_count                     0
total_energy_kwh                  0
hour                              0
day_of_week                       0
month                             0
is_weekend                        0
lag_1h_session_count              1
lag_24h_session_count            24
lag_168h_session_count          168
lag_1h_energy                     1
lag_24h_energy                   24
lag_168h_energy                 168
rolling_7d_avg_session_count    167
rolling_7d_avg_energy           167
dtype: int64

# **Train/Test Split**

Rows with incomplete lag/rolling history (the first 168 hours) are dropped. The remaining data is split chronologically, not randomly: the last 90 days are held out as the test set, with everything before used for training. This ensures models are only ever evaluated on genuinely unseen, future data, a critical requirement for time-series forecasting, since a random split would let models “see” future information during training and produce misleadingly good accuracy.

*   Training set: 23,975 hourly records (∼2.7 years)
*   Test set: 2,160 hourly records (90 days)

In [5]:
hourly_clean = hourly_demand.dropna()

cutoff_date = hourly_clean.index.max() - pd.Timedelta(days=90)
train = hourly_clean[hourly_clean.index <= cutoff_date]
test = hourly_clean[hourly_clean.index > cutoff_date]

print(train.shape, test.shape)

(23975, 14) (2160, 14)


In [6]:
# saving the train and test datasets
train.to_csv('/kaggle/working/train.csv')
test.to_csv('/kaggle/working/test.csv')